# Notebook 4 — Feature Engineering

## What we do
We **create new columns** (features) from the existing ones, to give the models more useful information.

## Example
A model cannot do much with `release_date = 2019-07-12`, but `release_year = 2019`, `release_month = 7` and `decade = 2010` are easy to use.

## The big rule: avoid Data Leakage
**Data leakage** = the model gets information it would **not have in real life** when making a prediction.

In Notebook 6 we will predict `high_engagement`, a column built from `vote_count`. So:
- `vote_count` must **not** be a feature (it *is* the answer).
- `popularity` and `vote_average` are computed from the votes → also too close to the answer.
- `revenue` is known only after the film is released → we exclude it too.

We also do **not** compute averages or medians here (they must be computed on the training set only, later, inside the pipeline).

In [1]:
import os

# The notebooks live in notebooks/, but the data folder is at the project root.
# This moves us to the project root so paths like "data/raw/..." always work.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

Working folder: c:\Users\ycode\Desktop\Projet\Machine Learning\CineMatch AI


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/processed/movies_clean.csv", parse_dates=["release_date"])
print(df.shape)

(2972, 14)


## 1. Genres and keywords back to lists
In the CSV they are written as `"Action|Comedy"`. We convert them back to lists.

In [3]:
def text_to_list(text):
    if pd.isna(text) or text == "":
        return []
    return text.split("|")

df["genres"] = df["genres"].apply(text_to_list)
df["keywords"] = df["keywords"].apply(text_to_list)
df[["title", "genres", "keywords"]].head()

,title,genres,keywords
0,Spider-Man: Brand New Day,"[Science Fiction, Action, Adventure]","[mind control, new york city, hero, mutation, ..."
1,Renegade Immortal: Battle of the Immortal Slayer,"[Animation, Action, Adventure, Fantasy]","[based on fairy tale, xianxia, based on tv ser..."
2,Resident Evil,"[Horror, Science Fiction, Adventure]","[outbreak, survival, decapitation, zombie, bas..."
3,The Odyssey,"[Adventure, Action, Fantasy]","[ship, trojan war, greek mythology, historical..."
4,The Fix,"[Action, Thriller]","[central intelligence agency (cia), war vetera..."


## 2. Date features
**Why**: habits change over time (streaming, cinema seasons...), and older films had more time to receive votes.

In [4]:
df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
df["release_decade"] = df["release_year"] // 10 * 10     # 2019 -> 2010
df["movie_age"] = 2026 - df["release_year"]              # years since release

df[["release_date", "release_year", "release_month", "release_decade", "movie_age"]].head()

,release_date,release_year,release_month,release_decade,movie_age
0,2026-07-29,2026,7,2020,0
1,2026-10-01,2026,10,2020,0
2,2026-09-16,2026,9,2020,0
3,2026-07-15,2026,7,2020,0
4,2026-09-10,2026,9,2020,0


## 3. Counting features
**Why**: a film described by many genres and keywords is easier to find on a platform; it is a rough measure of how "rich" the film description is.

In [5]:
df["n_genres"] = df["genres"].apply(len)
df["n_keywords"] = df["keywords"].apply(len)

df[["n_genres", "n_keywords"]].describe()

,n_genres,n_keywords
count,2972.000000,2972.000000
mean,2.715680,14.399058
std,1.000294,8.545731
min,0.000000,0.000000
25%,2.000000,8.750000
50%,3.000000,14.000000
75%,3.000000,19.000000
max,7.000000,98.000000


## 4. Runtime category
**Why**: the effect of the duration is not linear (a short film and a very long film behave differently), so categories can help.

In [6]:
def runtime_category(minutes):
    if pd.isna(minutes):
        return "unknown"
    if minutes < 90:
        return "short"
    if minutes < 120:
        return "standard"
    if minutes < 150:
        return "long"
    return "very_long"

df["runtime_category"] = df["runtime"].apply(runtime_category)
df["runtime_category"].value_counts()

runtime_category
standard     1720
long          701
short         369
very_long     159
unknown        23
Name: count, dtype: int64

## 5. Genre features
- `main_genre`: the first genre of the film.
- One 0/1 column per genre (a film can be Action **and** Comedy). We use a **fixed list** of genres, so nothing is learned from the data.

In [7]:
def get_main_genre(genres):
    if len(genres) == 0:
        return "Unknown"
    return genres[0]

df["main_genre"] = df["genres"].apply(get_main_genre)

GENRES = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
          "Family", "Fantasy", "Horror", "Romance", "Science Fiction", "Thriller"]

genre_columns = []
for genre in GENRES:
    column_name = "genre_" + genre.replace(" ", "_")
    df[column_name] = df["genres"].apply(lambda film_genres: 1 if genre in film_genres else 0)
    genre_columns.append(column_name)

df[["title", "main_genre"] + genre_columns[:4]].head()

,title,main_genre,genre_Action,genre_Adventure,genre_Animation,genre_Comedy
0,Spider-Man: Brand New Day,Science Fiction,1,1,0,0
1,Renegade Immortal: Battle of the Immortal Slayer,Animation,1,1,1,0
2,Resident Evil,Horror,0,1,0,0
3,The Odyssey,Adventure,1,1,0,0
4,The Fix,Action,1,0,0,0


## 6. Other simple indicators
- `has_budget`: do we know the budget? (0/1) — big productions often have a known budget.
- `is_english`: original language is English? (0/1)
- `overview_word_count`: length of the synopsis (0 if missing).

In [8]:
df["has_budget"] = df["budget"].notna().astype(int)
df["is_english"] = (df["original_language"] == "en").astype(int)
df["overview_word_count"] = df["overview"].fillna("").str.split().str.len()

df[["has_budget", "is_english", "overview_word_count"]].describe()

,has_budget,is_english,overview_word_count
count,2972.000000,2972.000000,2972.000000
mean,0.802153,0.865074,45.440444
std,0.398443,0.341702,21.172469
min,0.000000,0.000000,0.000000
25%,1.000000,1.000000,28.000000
50%,1.000000,1.000000,41.000000
75%,1.000000,1.000000,59.000000
max,1.000000,1.000000,157.000000


## 7. Choose the features and check there is no leakage
We write the list of columns the models are allowed to use, and a list of forbidden columns. The `assert` stops the notebook with an error if a forbidden column is used by mistake.

In [9]:
numeric_features = ["runtime", "budget", "release_year", "release_month", "movie_age",
                    "n_genres", "n_keywords", "overview_word_count"]
binary_features = ["has_budget", "is_english"] + genre_columns
categorical_features = ["main_genre", "runtime_category", "original_language"]
text_feature = "overview"                      # used in the TF-IDF notebook

forbidden = ["vote_count", "vote_average", "popularity", "revenue"]

all_features = numeric_features + binary_features + categorical_features
for column in all_features:
    assert column not in forbidden, column + " is forbidden (data leakage)!"
print("OK:", len(all_features), "features, no leakage")

OK: 26 features, no leakage


In [10]:
# Extra check: is any feature suspiciously linked to vote_count?
check = df[numeric_features].corrwith(np.log1p(df["vote_count"]))
print(check.sort_values(ascending=False).round(2))

n_keywords             0.50
budget                 0.34
runtime                0.29
movie_age              0.22
n_genres               0.20
release_month          0.05
overview_word_count    0.02
release_year          -0.22
dtype: float64


**Reading the result**: a correlation close to 1 would be suspicious. A moderate one (for example `movie_age` or `has_budget`) is normal and useful.

Note that `budget` and `runtime` still contain missing values. That is fine: the pipeline will fill them later, **using the training set only**.

## 8. Save

In [11]:
df_save = df.copy()
df_save["genres"] = df_save["genres"].apply(lambda items: "|".join(items))
df_save["keywords"] = df_save["keywords"].apply(lambda items: "|".join(items))
df_save.to_csv("data/processed/movies_features.csv", index=False)
print("Saved:", df_save.shape)
df_save.head()

Saved: (2972, 37)


,movie_id,title,overview,release_date,runtime,original_language,genres,keywords,budget,revenue,...,genre_Drama,genre_Family,genre_Fantasy,genre_Horror,genre_Romance,genre_Science_Fiction,genre_Thriller,has_budget,is_english,overview_word_count
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,2026-07-29,145.0,en,Science Fiction|Action|Adventure,mind control|new york city|hero|mutation|secre...,225000000.0,2.505477e+09,...,0,0,0,0,0,1,0,1,1,68
1,1599191,Renegade Immortal: Battle of the Immortal Slayer,Trapped in a fatal pursuit within the Celestia...,2026-10-01,NaN,zh,Animation|Action|Adventure|Fantasy,based on fairy tale|xianxia|based on tv series...,NaN,NaN,...,0,0,1,0,0,0,0,0,0,39
2,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,2026-09-16,95.0,en,Horror|Science Fiction|Adventure,outbreak|survival|decapitation|zombie|based on...,75000000.0,2.454000e+08,...,0,0,0,1,0,1,0,1,1,19
3,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",2026-07-15,173.0,en,Adventure|Action|Fantasy,ship|trojan war|greek mythology|historical fic...,250000000.0,1.767464e+09,...,0,0,1,0,0,0,0,1,1,46
4,1228834,The Fix,Disillusioned by the end of the war in Afghani...,2026-09-10,142.0,en,Action|Thriller,central intelligence agency (cia)|war veteran|...,32000000.0,NaN,...,0,0,0,0,0,0,1,1,1,26


## What to remember (for the oral)
- **Feature engineering** = creating useful columns from existing ones. For each one, you must be able to say *why*.
- **Data leakage** = using information the model could not know at prediction time. The column used to build the target (`vote_count`) is never a feature.
- We create features with simple, fixed rules (no averages learned from the data) so there is no leakage.
- Next: TF-IDF on the `overview` text (Notebook 5).